# **Preprocessing**

In [7]:
# Import libraries
from pathlib import Path
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline

In [9]:
# Loading the data
input_path = Path("..") / 'data' / 'interim' / 'dataset_merged.csv'
df = pd.read_csv(input_path)
display(df.head(5))
print("Shape of the piezometer dataframe:\n", df.shape)

,Unnamed: 0_x,date_index,latitude,longitude,sunrise,sunset,daylight_duration,precipitation_sum,shortwave_radiation_sum,et0_fao_evapotranspiration,...,pressure_msl_mean,wind_speed_10m_mean,soil_moisture_0_to_100cm_mean,soil_temperature_0_to_100cm_mean,code_bss,Unnamed: 0_y,bss_id,niveau_nappe_eau,mode_obtention,nom_producteur
0,0,2017-01-01,42.685696,2.685378,2017-01-01T07:19,2017-01-01T16:26,32800.70,0.0,7.88,0.95,...,1024.5,5.3,0.244,9.2,10906X0039/C2-1,5438,BSS002MNYD,101.85,Valeur reconstituée,Syndicat Mixte pour la Protection et la Gestio...
1,1,2017-01-02,42.685696,2.685378,2017-01-02T07:19,2017-01-02T16:27,32850.58,0.0,4.29,0.56,...,1022.0,4.6,0.244,9.2,10906X0039/C2-1,5439,BSS002MNYD,101.89,Valeur reconstituée,Syndicat Mixte pour la Protection et la Gestio...
2,2,2017-01-03,42.685696,2.685378,2017-01-03T07:19,2017-01-03T16:28,32904.28,0.0,8.02,1.59,...,1023.5,11.8,0.244,8.9,10906X0039/C2-1,5440,BSS002MNYD,101.98,Valeur reconstituée,Syndicat Mixte pour la Protection et la Gestio...
3,3,2017-01-04,42.685696,2.685378,2017-01-04T07:19,2017-01-04T16:29,32961.75,0.0,8.08,2.52,...,1022.9,25.2,0.244,8.8,10906X0039/C2-1,5441,BSS002MNYD,101.98,Valeur reconstituée,Syndicat Mixte pour la Protection et la Gestio...
4,4,2017-01-05,42.685696,2.685378,2017-01-05T07:19,2017-01-05T16:30,33022.91,0.0,6.93,2.07,...,1026.5,30.5,0.243,8.7,10906X0039/C2-1,5442,BSS002MNYD,102.01,Valeur reconstituée,Syndicat Mixte pour la Protection et la Gestio...


Shape of the piezometer dataframe:
 (3405, 21)


In [ ]:
def preprocessing(df):
    """
    This function performed the following steps:
    1. preprocessing preparation (dropping undesire columns; splitting the target 'y' and the explicatives features 'X')
    2. Temporal split (X_train; X_test; y_train & y_test)
    At the end, this function return the different datasets to perform Machine Learning algorithms.
    """
    # Setting 'date_index' as index in the dataframe
    df = df.set_index("date_index")
    df.index = pd.to_datetime(df.index)
    
    # Dropping useless columns
    columns_to_drop = ['Unnamed: 0_x', 'code_bss', 'bss_id', 'mode_obtention', 'nom_producteur', 'latitude', 'longitude']
    df = df.drop(columns = columns_to_drop)
    
    # Temporal split (80/20% for the train/test sets split)
    split_idx = int(len(df) * 0.8)
    df_train = df.iloc[:split_idx]
    df_test = df.iloc[split_idx:]
    
    y_train = df_train['niveau_nappe_eau']
    X_train = df_train.drop(columns='niveau_nappe_eau')
   
    y_test = df_test['niveau_nappe_eau']
    X_test = df_test.drop(columns='niveau_nappe_eau')
    
    return X_train, X_test, y_train, y_test

In [25]:
X_train, X_test, y_train, y_test = preprocessing(df)
display(X_train)

,sunrise,sunset,daylight_duration,precipitation_sum,shortwave_radiation_sum,et0_fao_evapotranspiration,cloud_cover_mean,pressure_msl_mean,wind_speed_10m_mean,soil_moisture_0_to_100cm_mean,soil_temperature_0_to_100cm_mean,Unnamed: 0_y
date_index,,,,,,,,,,,,
2017-01-01,2017-01-01T07:19,2017-01-01T16:26,32800.70,0.0,7.88,0.95,47,1024.5,5.3,0.244,9.2,5438
2017-01-02,2017-01-02T07:19,2017-01-02T16:27,32850.58,0.0,4.29,0.56,93,1022.0,4.6,0.244,9.2,5439
2017-01-03,2017-01-03T07:19,2017-01-03T16:28,32904.28,0.0,8.02,1.59,3,1023.5,11.8,0.244,8.9,5440
2017-01-04,2017-01-04T07:19,2017-01-04T16:29,32961.75,0.0,8.08,2.52,5,1022.9,25.2,0.244,8.8,5441
2017-01-05,2017-01-05T07:19,2017-01-05T16:30,33022.91,0.0,6.93,2.07,14,1026.5,30.5,0.243,8.7,5442
...,...,...,...,...,...,...,...,...,...,...,...,...
2024-07-11,2024-07-11T04:22,2024-07-11T19:27,54343.97,0.0,26.27,6.97,12,1014.6,13.5,0.134,23.4,8157
2024-07-12,2024-07-12T04:22,2024-07-12T19:27,54265.74,0.3,14.97,5.10,68,1014.9,23.7,0.134,23.4,8158
2024-07-13,2024-07-13T04:23,2024-07-13T19:26,54183.69,0.0,24.09,5.53,61,1013.8,9.0,0.134,23.2,8159


# **Machine Learning**

## Linear regression

In [11]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

In [26]:
def linear_regression(df):
    """
    Function to run a Linear Regression on a train and test set.
    Return the prediction and evaluation metrics (R2 on train and test, RMSE, MAE)
    """
    model = LinearRegression()
    X_train, X_test, y_train, y_test = preprocessing(df)
    
    # Training of the model
    model.fit(X_train, y_train)
    
    # Prediction
    y_pred = model.predict(X_test)
    
    # Evaluation metrics
    r2_train = model.score(X_train, y_train)
    r2_test = model.score(X_test, y_test)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    mae = mean_absolute_error(y_test, y_pred)
    
    evaluation_metrics_lr = {
        'R2_train': r2_train,
        'R2_test': r2_test,
        'RMSE': rmse,
        'MAE': mae}
    
    return y_pred, evaluation_metrics_lr

In [27]:
y_pred, evaluation_metrics_lr = linear_regression(df)

ValueError: could not convert string to float: '2017-01-01T07:19'

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 8))

# Série complète
axes[0].plot(df.index, df["niveau_nappe_eau"])
axes[0].set_title("Évolution du niveau de nappe eau")

# Prédictions vs réel
axes[1].plot(y_test.index, y_test, label="Réel")
axes[1].plot(y_test.index, y_pred, label="Prédit")
axes[1].set_title("y_test vs y_pred")
axes[1].legend()

plt.tight_layout()
plt.show()

## XGBoost regressor

In [21]:
from xgboost import XGBRegressor
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

In [ ]:
def xgboost_regressor(df, n_estimators=100, learning_rate=0.1, max_depth=4, 
                      subsample=0.8, colsample_bytree=0.8, random_state=42):
    """
    Function to run a XGBoost Regressor on a train and test set.
    Return the prediction and evaluation metrics (R2 on train and test, RMSE, MAE)
    """
    model = XGBRegressor(
        n_estimators        = n_estimators,
        learning_rate       = learning_rate,
        max_depth           = max_depth,
        subsample           = subsample,
        colsample_bytree    = colsample_bytree,
        random_state        = random_state,
        )
    
    X_train, X_test, y_train, y_test = preprocessing(df)
    
    # Training of the model
    model.fit(X_train, y_train)
    
    # Prediction
    y_pred = model.predict(X_test)
    
    # Evaluation metrics
    r2_train = model.score(X_train, y_train)
    r2_test = model.score(X_test, y_test)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    mae = mean_absolute_error(y_test, y_pred)
    
    evaluation_metrics_xgboost_reg = {
        'R2_train': r2_train,
        'R2_test': r2_test,
        'RMSE': rmse,
        'MAE': mae}
    
    return y_pred, evaluation_metrics_xgboost_reg

In [ ]:
y_pred, evaluation_metrics_xgboost_reg = xgboost_regressor(df)

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 8))

# Série complète
axes[0].plot(df.index, df["niveau_nappe_eau"])
axes[0].set_title("Évolution du niveau de nappe eau")

# Prédictions vs réel
axes[1].plot(y_test.index, y_test, label="Réel")
axes[1].plot(y_test.index, y_pred, label="Prédit")
axes[1].set_title("y_test vs y_pred")
axes[1].legend()

plt.tight_layout()
plt.show()